# Clairvue — Management Claim & Peer Evidence Assistant

Clairvue is a RAG system that validates bank management statements (e.g. earnings-call claims like *"Consumer credit remains resilient"*) against formal SEC disclosures, quantitative XBRL metrics, and peer-bank evidence. Given a claim, it decomposes the claim into atomic, independently verifiable assertions, retrieves grounded evidence for each one, and returns a structured assessment — **supported / partially supported / contradicted / insufficient evidence** — with citations back to the actual filings and metrics. It also supports multi-bank peer comparison on a given risk theme.

This notebook is **for demo purpose only**. All embeddings and generation run exclusively on Mistral models (`mistral-embed` + a chat model), with no other model provider anywhere in the pipeline.

**Scope:** Prototype uses public SEC filings from JPMorgan Chase, Bank of America, and Citigroup, Q4 2022 through Q4 2023. The risk theme covered is consumer credit quality (charge-offs, delinquencies, provisions, "normalization" language).

Run the cells below in order. The first cell clones the repo and installs dependencies; the data (chunked filings, embeddings, ChromaDB index, and curated metrics) is expected to already exist in Google Drive at `MyDrive/clairvue_data` (built ahead of time via `scripts/download_filings.py` + `scripts/build_index.py`).

In [1]:
# Install dependencies
import subprocess, sys

!git clone https://github.com/dannyyuanxu/clairvue.git 2>/dev/null || echo "already cloned"
%cd clairvue

# Run pip quietly and only surface output if it actually fails. The "ERROR: pip's
# dependency resolver..." lines you may have seen are NOT failures -- they're pre-existing
# conflicts among Colab's own pre-installed packages (google-adk / opentelemetry-exporter-*),
# none of which Clairvue uses. The install itself succeeds, so we hide that noise here.
print("Installing dependencies (~1 min)...")
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True,
)
if _pip.returncode != 0:
    print(_pip.stdout)
    print(_pip.stderr)
    raise RuntimeError("pip install failed -- see output above")
print("Dependencies installed.")

# Mount Google Drive (data lives here). A brand-new Colab runtime always prompts for auth
# once (the VM is ephemeral and doesn't persist the token) -- to avoid re-auth, use
# Runtime > "Restart session" rather than "Disconnect and delete runtime". force_remount=False
# makes re-running this cell within the same session a no-op (no second popup).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Load API key from Colab Secrets (Settings -> Secrets -> add MISTRAL_API_KEY)
import os
from google.colab import userdata

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
os.environ["EMBEDDING_MODEL"] = "mistral-embed"
# Medium for demo quality. (ENRICHMENT_MODEL is intentionally not set here -- chunk enrichment
# runs only during the local index build, never at demo/query time, so it has no effect in Colab.)
os.environ["LLM_MODEL"] = "mistral-medium-3-5"


# Point to Drive data
DATA_DIR = "/content/drive/MyDrive/clairvue_data"
os.environ["CHROMA_PERSIST_DIR"] = f"{DATA_DIR}/chroma_db"

print("Setup complete.")

already cloned
/content/clairvue
Installing dependencies (~1 min)...
Dependencies installed.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.


In [2]:
print('Checking for latest changes from the repository...')
!git pull -q
print('Repository updated.')

Checking for latest changes from the repository...
Repository updated.


In [3]:
import sys
sys.path.insert(0, '/content/clairvue')

from src.vector_store import load_collection
from src.retrieval import EvidenceRetriever

collection = load_collection()
print(f"Loaded ChromaDB collection: {collection.count()} chunks indexed")

Loaded ChromaDB collection: 9750 chunks indexed


## Demo 1 — Claim Validation

Test whether a management statement is supported by formal filing evidence.

In [4]:
from src.generation import answer_claim
from src.demo import display_claim_assessment

statement = "Consumer credit remains resilient, and losses are normalizing in line with expectations."
result = answer_claim(statement, ticker="JPM", verbose=False)
display_claim_assessment(result, verbose=False) # To see analyst follow-up questions: set verbose=True in both calls

Decomposed into 3 atomic claims:
  - Consumer credit is currently resilient.
  - Consumer credit losses are normalizing.
  - The normalization of consumer credit losses aligns with prior expectations.


## Statement assessment
> Consumer credit remains resilient, and losses are normalizing in line with expectations.

**Overall: 🟡 Partially supported**

---
### 🟡 Consumer credit is currently resilient.
**Partially supported**

JPMorgan Chase's net charge-off rate (0.49% in 2023-Q4) is rising from the pandemic trough (0.24%) but remains below its 2019 baseline (0.73%), indicating credit losses are normalizing rather than deteriorating. However, the rate is rising sequentially, and the provision for credit losses increased significantly (YoY +45.9%), which introduces uncertainty. Peer banks show a mixed picture: Citigroup's net charge-off rate (1.18%) is higher and rising, while Bank of America's (0.37%) is lower and flat, suggesting divergence in consumer credit resilience across peers.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate | 2023-Q4 | 0.49% | QoQ +0.06pp (rising from trough 0.24%), below 2019 baseline (0.73%) by 0.24pp; peer cross-section: JPM 0.49% < C 1.18%, > BAC 0.37% |
| JPM | provision_for_credit_losses | 2023-Q4 | \$9.32B | YoY +45.9% |
| JPM | allowance_for_credit_losses | 2023-Q4 | \$22.42B | YoY +13.7% |
| BAC | net_charge_off_rate | 2023-Q4 | 0.37% | QoQ +0.01pp (flat), no pre-2020 baseline available |
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available |

**Supporting**
- "The Firm’s focus is on serving primarily the prime segment of the consumer credit market." — *JPMorgan Chase, 10-K 2023-Q4*

**Qualifying / watch**
- "30+ days past due to total retained loans (auto and other), 2023-Q1: 0.88%" — *JPMorgan Chase, 10-Q 2023-Q1*
- "30+ days past due to total retained loans (auto and other), 2023-Q2: 0.68%" — *JPMorgan Chase, 10-Q 2023-Q2*
- "30+ days past due to total retained loans (auto and other), 2023-Q3: 0.82%" — *JPMorgan Chase, 10-Q 2023-Q3*

**Missing**
- Pre-2020 baseline net charge-off rates for Bank of America and Citigroup to fully assess normalization
- Explicit delinquency rate trends for JPMorgan Chase's residential real estate and credit card portfolios in 2023-Q4
- Management's definition of 'resilient' and any internal thresholds or guidance for consumer credit performance

---
### 🟡 Consumer credit losses are normalizing.
**Partially supported**

JPMorgan Chase's net charge-off rate for consumer credit (0.49% in 2023-Q4) is rising from the pandemic trough (0.24%) but remains below its 2019 baseline (0.73%), which is consistent with normalization. However, the claim is only partially supported because the rate has not yet returned to the pre-pandemic baseline. Peer banks' net charge-off rates (BAC: 0.37%, C: 1.18%) show divergence, with Citigroup's rate rising above JPMorgan's and Bank of America's rate remaining lower, but baselines for peers are missing, limiting comparability.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate | 2023-Q4 | 0.49% | QoQ +0.06pp (rising), vs 2019 baseline 0.73%: below by 0.24pp (not yet back to pre-pandemic normal) |
| BAC | net_charge_off_rate | 2023-Q4 | 0.37% | QoQ +0.01pp (flat), no pre-2020 baseline available |
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available |

**Supporting**
- "Net charge-off rates | 0.17 | 2.45 | 0.14 | 0.52 | 0.09 | 1.47 | 0.03 | 0.27" — *JPMorgan Chase, 10-K 2023-Q4*

**Qualifying / watch**
- "The provision for credit losses was \$2.3 billion, reflecting a net addition of \$1.1 billion to the allowance for credit losses and \$1.1 billion of net charge-offs." — *JPMorgan Chase, 10-Q 2023-Q1*

**Missing**
- Pre-2020 baseline net charge-off rates for Bank of America and Citigroup to confirm normalization trend
- Explicit definition of 'normalization' in JPMorgan Chase's filings

---
### ⚪ The normalization of consumer credit losses aligns with prior expectations.
**Insufficient evidence**

The primary evidence does not contain any explicit prior expectations or guidance from JPMorgan Chase regarding the normalization of consumer credit losses, so the claim that losses align with prior expectations cannot be verified. The net charge-off rate for JPMorgan Chase is rising (0.49% in 2023-Q4, +0.06pp QoQ) but remains below the 2019 baseline (0.73%), which is consistent with normalization, but the absence of prior expectations in the evidence prevents confirmation of alignment. Peer banks' evidence shows varying trends but cannot substitute for JPMorgan's own guidance.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate (annualized) | 2023-Q4 | 0.49% | QoQ +0.06pp (rising), vs 2019 baseline 0.73%: below by 0.24pp (not yet back to pre-pandemic normal) |

**Qualifying / watch**
- "The provision for credit losses was \$2.3 billion, reflecting a net addition of \$1.1 billion to the allowance for credit losses and \$1.1 billion of net charge-offs." — *JPMorgan Chase, 10-Q 2023-Q1*

**Missing**
- Explicit prior expectations or guidance from JPMorgan Chase regarding the normalization of consumer credit losses

## Demo 2 — Peer Comparison

Compare consumer-credit risk evidence across JPMorgan, Bank of America, and Citigroup.

In [5]:
from src.generation import compare_peers
from src.demo import display_peer_comparison

question = "Which of JPMorgan, Bank of America, and Citigroup shows the strongest evidence of increasing consumer-credit pressure in 2023?"
result = compare_peers(question, risk_theme="consumer_credit")
display_peer_comparison(result)

## Peer comparison
> Which of JPMorgan, Bank of America, and Citigroup shows the strongest evidence of increasing consumer-credit pressure in 2023?

---
### JPM — JPMorgan Chase 🔴 Deteriorating
JPM shows rising net charge-off rate (0.49% in 2023-Q4, +0.06pp QoQ) and significant YoY increases in provision for credit losses (+45.9%) and allowance for credit losses (+13.7%). However, the net charge-off rate remains below pre-pandemic baseline (0.73%). Evidence of loan growth (YoY +17.6%) but no explicit narrative on consumer-credit pressure in 2023 filings.

*Metrics:* net_charge_off_rate: 0.49% (2023-Q4, +0.06pp QoQ), provision_for_credit_losses: \$9.32B (YoY +45.9%), allowance_for_credit_losses: \$22.42B (YoY +13.7%), total_loans: \$1258.45B (YoY +17.6%)

**Evidence**
- JPMorgan Chase | Type: regulatory_filing | Filing: 10-K 2023-Q4 | Section: Consumer, excluding credit card | Authority: formal_disclosure: 'Loans increased from December 31, 2022 driven by residential real estate loans associated with First Republic and higher auto loans.'
- JPMorgan Chase | Type: regulatory_filing | Filing: 10-Q 2023-Q1 | Section: CONSUMER CREDIT PORTFOLIO | Authority: formal_disclosure: 'The Firm’s retained consumer portfolio consists primarily of loans and lending-related commitments for residential real estate, credit card, scored auto and business banking.'

---
### BAC — Bank of America 🔴 Deteriorating
BAC shows a modest rise in net charge-off rate (0.37% in 2023-Q4, +0.01pp QoQ) and a sharp YoY increase in provision for credit losses (+92.1%). However, the net charge-off rate remains low and flat QoQ. Filings note higher net credit losses driving lower risk-adjusted margin in 2023-Q1, but no explicit baseline for pre-pandemic normalization.

*Metrics:* net_charge_off_rate: 0.37% (2023-Q4, +0.01pp QoQ), provision_for_credit_losses: \$4.72B (YoY +92.1%), allowance_for_credit_losses: \$13.34B (YoY +5.2%), total_loans: \$1040.39B (YoY +0.7%)

**Evidence**
- Bank of America | Type: regulatory_filing | Filing: 10-Q 2023-Q1 | Section: Consumer Lending | Authority: formal_disclosure: 'During the three months ended March 31, 2023, the total risk-adjusted margin decreased 171 bps compared to the same period in 2022 driven by lower net interest margin, lower fee income and higher net credit losses.'
- Bank of America | Type: regulatory_filing | Filing: 10-Q 2023-Q3 | Section: Credit Card and Other Consumer | Authority: formal_disclosure: 'The provision for credit losses for the current-year periods was driven by the Corporation’s consumer portfolio primarily due to credit card loan growth and asset quality.'

---
### C — Citigroup 🔴 Deteriorating
C shows the strongest evidence of increasing consumer-credit pressure: net charge-off rate rose to 1.18% in 2023-Q4 (+0.17pp QoQ), the highest among peers. Provision for credit losses increased YoY by 64.1%, and filings explicitly state that net credit losses in the cards businesses are expected to reach pre-pandemic levels by the end of 2023. No pre-2020 baseline is provided, but the trend is clearly deteriorating.

*Metrics:* net_charge_off_rate: 1.18% (2023-Q4, +0.17pp QoQ), provision_for_credit_losses: \$7.79B (YoY +64.1%), allowance_for_credit_losses: \$18.14B (YoY +6.9%), total_loans: \$671.22B (YoY +4.8%)

**Evidence**
- Citigroup | Type: regulatory_filing | Filing: 10-Q 2023-Q3 | Section: AND RESULTS OF OPERATIONS | Authority: formal_disclosure: 'Net credit losses in the cards businesses are expected to reach pre-pandemic levels by the end of 2023.'

---
**Strongest deterioration signal:** Citigroup (C) shows the strongest evidence of increasing consumer-credit pressure in 2023, with the highest net charge-off rate (1.18% in 2023-Q4, +0.17pp QoQ) and explicit narrative in filings about net credit losses in cards businesses reaching pre-pandemic levels by end-2023.

**Overall:** Citigroup (C) exhibits the strongest evidence of increasing consumer-credit pressure in 2023, driven by the highest net charge-off rate (1.18% in 2023-Q4, +0.17pp QoQ) and explicit narrative about normalization to pre-pandemic levels. JPMorgan (JPM) also shows deterioration but remains below its pre-pandemic baseline. Bank of America (BAC) shows the weakest signal among the three, with a low and relatively flat net charge-off rate.

**Comparability limitations**
- Net charge-off rate baselines for pre-pandemic normalization are only available for JPM (0.73% in 2019); BAC and C lack comparable pre-2020 baselines in the supplied evidence.
- Filing types and sections differ across banks (e.g., JPM's 10-K vs. BAC's 10-Q), which may limit direct comparability of narrative evidence.
- Total loan growth varies significantly (JPM +17.6% YoY, BAC +0.7% YoY, C +4.8% YoY), which may affect the interpretation of charge-off rates and provisions.
- Provision for credit losses and allowance for credit losses are not directly comparable due to differing portfolio compositions and accounting policies.

## Demo 3 — Appropriate Abstention

Test that the system correctly abstains when evidence is insufficient to support a definitive claim.

In [6]:
from src.demo import display_claim_assessment

result3 = answer_claim(
    "Consumer credit deterioration in 2023 was concentrated among younger, first-time borrowers.",
    risk_theme="consumer_credit",
)
display_claim_assessment(result3)

Decomposed into 3 atomic claims:
  - Consumer credit deterioration occurred in 2023
  - The deterioration in consumer credit in 2023 was concentrated among younger borrowers
  - The deterioration in consumer credit in 2023 was concentrated among first-time borrowers


## Statement assessment
> Consumer credit deterioration in 2023 was concentrated among younger, first-time borrowers.

**Overall: 🟡 Partially supported**

---
### 🟡 Consumer credit deterioration occurred in 2023
**Partially supported**

For Bank of America (BAC), the net charge-off rate rose from 0.32% in 2023-Q1 to 0.37% in 2023-Q4, indicating a rising trend in credit losses. However, the absence of a pre-2020 baseline for BAC means we cannot confirm whether this rise represents deterioration relative to historical norms. Peer context shows BAC's net charge-off rate (0.37%) is the lowest among peers (JPM: 0.49%, C: 1.18%), suggesting BAC's credit performance is relatively strong. The claim is partially supported by the rising trend in BAC's charge-offs but lacks confirmation of deterioration relative to pre-pandemic levels.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| BAC | net_charge_off_rate | 2023-Q4 | 0.37% | QoQ +0.01pp (flat), rising from 0.32% in 2023-Q1 to 0.37% in 2023-Q4 |
| BAC | provision_for_credit_losses | 2023-Q4 | \$4.72B | YoY +92.1% |
| BAC | allowance_for_credit_losses | 2023-Q4 | \$13.34B | YoY +5.2% |

**Qualifying / watch**
- "BAC net_charge_off_rate (annualized loss rate): latest 2023-Q4: 0.37% | QoQ +0.01pp (flat)" — *Bank of America, 10-K/10-Q 2023-Q4*

**Peer context**
- "JPM net_charge_off_rate (annualized loss rate): latest 2023-Q4: 0.49% | QoQ +0.06pp (rising) | vs 2019 baseline 0.73%: below by 0.24pp" — *JPMorgan Chase, 10-K/10-Q 2023-Q4*
- "C net_charge_off_rate (annualized loss rate): latest 2023-Q4: 1.18% | QoQ +0.17pp (rising)" — *Citigroup, 10-K/10-Q 2023-Q4*

**Missing**
- Pre-2020 baseline for Bank of America's net charge-off rate to confirm whether the 2023 rise represents deterioration relative to historical norms

---
### ⚪ The deterioration in consumer credit in 2023 was concentrated among younger borrowers
**Insufficient evidence**

The provided evidence does not contain any age-based segmentation of credit deterioration, delinquency rates, or charge-off rates. The filings and metrics focus on aggregate consumer credit performance (e.g., net charge-off rates, provisions, allowances) without breaking down trends by borrower age. Without explicit evidence on younger borrowers, the claim cannot be verified.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate | 2023-Q4 | 0.49% | QoQ +0.06pp (rising), below 2019 baseline (0.73%) by 0.24pp |
| BAC | net_charge_off_rate | 2023-Q4 | 0.37% | QoQ +0.01pp (flat), no pre-2020 baseline available |
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available |

**Missing**
- Age-based segmentation of credit deterioration (e.g., delinquency or charge-off rates by borrower age cohort)
- Pre-2020 baseline for BAC and C net charge-off rates to assess normalization

---
### ⚪ The deterioration in consumer credit in 2023 was concentrated among first-time borrowers
**Insufficient evidence**

None of the provided formal disclosures or metrics explicitly address the concentration of credit deterioration by borrower type (e.g., first-time vs. repeat borrowers). The evidence includes high-level credit metrics (e.g., net charge-off rates) and general portfolio descriptions but lacks granular segmentation by borrower experience or vintage.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate | 2023-Q4 | 0.49% | QoQ +0.06pp (rising), vs 2019 baseline 0.73%: below by 0.24pp (not yet back to pre-pandemic normal) |
| BAC | net_charge_off_rate | 2023-Q4 | 0.37% | QoQ +0.01pp (flat), no pre-2020 baseline available |
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available |

**Missing**
- Segmentation of credit deterioration by borrower type (e.g., first-time vs. repeat borrowers) in formal disclosures
- Vintage analysis or cohort performance data for consumer credit portfolios
- Explicit statements linking deterioration to first-time borrowers in 10-K/10-Q filings

## Demo 4 — Real BAC comments made in Media (Brian Moynihan, AP news article, 2023-10-17)


In [8]:
# check back on past management statements
statement = (
"[in late 2023] We did this in a healthy but slowing economy that saw US consumer spending still ahead of last year but continuing to slow."
)

# Without verbose: the core verdict, rationale, metrics, and evidence per claim.
result4 = answer_claim(statement, ticker="BAC", verbose=True)
display_claim_assessment(result4, verbose=False)

Decomposed into 4 atomic claims:
  - The US economy was healthy in late 2023
  - The US economy was slowing in late 2023
  - US consumer spending in late 2023 was higher than in the same period of the previous year
  - US consumer spending was continuing to slow in late 2023


## Statement assessment
> [in late 2023] We did this in a healthy but slowing economy that saw US consumer spending still ahead of last year but continuing to slow.

**Overall: 🟡 Partially supported**

---
### 🟢 The US economy was healthy in late 2023
**Supported**

Bank of America's 10-K 2023-Q4 explicitly states that the latest consensus estimates for Q4 2023 included a 3.9% unemployment rate and 2.6% YoY GDP growth, which it describes as reflecting a 'tighter labor market and healthy growth.' Peer banks' credit metrics (e.g., JPM's net charge-off rate rising but still below 2019 baseline) align with a picture of economic health, though they are not the primary basis for this assessment.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| JPM | net_charge_off_rate | 2023-Q4 | 0.49% | QoQ +0.06pp (rising), vs 2019 baseline 0.73%: below by 0.24pp (not yet back to pre-pandemic normal) |

**Supporting**
- "As of December 31, 2023, the latest consensus estimates for the U.S. average unemployment rate for the fourth quarter of 2023 was 3.9 percent and U.S. GDP was forecasted to grow 2.6 percent year-over-year in the fourth quarter of 2023, reflecting a tighter labor market and healthy growth compared to our macroeconomic outlook as of December 31, 2022." — *Bank of America, 10-K 2023-Q4*

**Missing**
- Actual realized GDP growth and unemployment rate for Q4 2023 (only consensus estimates are provided)
- Pre-2020 baseline for Bank of America's net charge-off rate to assess normalization

---
### 🔴 The US economy was slowing in late 2023
**Contradicted**

Bank of America's 10-K 2023-Q4 explicitly states that the latest consensus estimates for Q4 2023 included U.S. GDP growth of 2.6% year-over-year, which reflects 'healthy growth' compared to its prior outlook. This directly contradicts the claim of a slowing economy. Peer banks' evidence does not provide direct macroeconomic growth figures for late 2023, so it neither aligns nor diverges from the primary bank's picture on this specific claim.

**Contradicting**
- "U.S. GDP was forecasted to grow 2.6 percent year-over-year in the fourth quarter of 2023, reflecting a tighter labor market and healthy growth compared to our macroeconomic outlook as of December 31, 2022." — *Bank of America, 10-K 2023-Q4*

**Missing**
- Actual realized GDP growth for Q4 2023 (consensus estimates are not realized data)
- Direct macroeconomic indicators (e.g., GDP, unemployment) from peer banks for late 2023

---
### ⚪ US consumer spending in late 2023 was higher than in the same period of the previous year
**Insufficient evidence**

The primary evidence from Bank of America's filings does not contain any direct or indirect statement about US consumer spending levels in late 2023 or their year-over-year comparison. The peer bank context also lacks explicit consumer spending data. The peer banks' evidence does not align or diverge from the primary bank's picture because no relevant evidence exists in either.

**Missing**
- Direct or indirect data on US consumer spending levels in late 2023 from Bank of America's filings
- Year-over-year comparison of US consumer spending for late 2023 vs. late 2022 from any provided evidence

---
### ⚪ US consumer spending was continuing to slow in late 2023
**Insufficient evidence**

The primary evidence from Bank of America's filings does not contain any direct or indirect statements about US consumer spending trends in late 2023. The provided excerpts focus on GDP growth estimates, credit loss allowances, and macroeconomic scenario variables, but none address consumer spending. Peer bank evidence also lacks direct references to consumer spending trends. The XBRL metrics provided (e.g., net charge-off rates, provisions, allowances) are credit-related and do not measure or imply consumer spending behavior.

**Missing**
- Direct or indirect statements about US consumer spending trends in late 2023 from Bank of America's filings
- Consumer spending metrics (e.g., retail sales, card spend volumes) for late 2023 from any source

## Demo 5 — Question based on Citi annual report


In [9]:
# check Citi quotes from annual report

statement5 = (
"Citi reported that fourth-quarter 2023 net credit loss rates and 90+ days past due delinquency rates increased quarter-over-quarter and year-over-year in both Branded Cards and Retail Services"
)
# Without verbose: the core verdict, rationale, metrics, and evidence per claim.
result5 = answer_claim(statement5, ticker="C", verbose=True)
display_claim_assessment(result5, verbose=False)

Decomposed into 5 atomic claims:
  - Citi reported fourth-quarter 2023 net credit loss rates increased quarter-over-quarter in Branded Cards
  - Citi reported fourth-quarter 2023 net credit loss rates increased year-over-year in Branded Cards
  - Citi reported fourth-quarter 2023 90+ days past due delinquency rates increased quarter-over-quarter in Branded Cards
  - Citi reported fourth-quarter 2023 90+ days past due delinquency rates increased year-over-year in Branded Cards
  - Citi reported fourth-quarter 2023 net credit loss rates and 90+ days past due delinquency rates increased quarter-over-quarter and year-over-year in Retail Services


## Statement assessment
> Citi reported that fourth-quarter 2023 net credit loss rates and 90+ days past due delinquency rates increased quarter-over-quarter and year-over-year in both Branded Cards and Retail Services

**Overall: 🟡 Partially supported**

---
### 🟢 Citi reported fourth-quarter 2023 net credit loss rates increased quarter-over-quarter in Branded Cards
**Supported**

Citi's 10-K for 2023-Q4 explicitly states that the fourth quarter of 2023 net credit loss rate in Branded Cards increased quarter-over-quarter. The XBRL metric for Citi's net_charge_off_rate confirms a QoQ increase of +0.17pp in 2023-Q4. Peer banks' evidence shows rising or flat net charge-off rates, which aligns with Citi's trend of increasing loss rates.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| C | net_charge_off_rate (annualized loss rate) | 2023-Q4 | 1.18% | QoQ +0.17pp (rising) |

**Supporting**
- "the fourth quarter of 2023 net credit loss rate and 90+ days past due delinquency rate in Branded Cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-K 2023-Q4*

---
### ⚪ Citi reported fourth-quarter 2023 net credit loss rates increased year-over-year in Branded Cards
**Insufficient evidence**

The primary evidence does not include a direct statement or data for Citi's 2023-Q4 Branded Cards net credit loss rate year-over-year change. While prior quarters (2023-Q1, Q2, Q3) explicitly state year-over-year increases, the 2023-Q4 10-K excerpt provided does not address Branded Cards net credit loss rates. Peer banks' evidence shows rising net charge-off rates but cannot substitute for Citi's own disclosure. Peer banks' evidence aligns with a rising trend in net charge-off rates, but this does not confirm Citi's specific claim.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| C | net_charge_off_rate (annualized loss rate) | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available for this bank |

**Qualifying / watch**
- "the third quarter of 2023 net credit loss rate and 90+ days past due delinquency rate in Branded cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-Q 2023-Q3*
- "the second quarter of 2023 net credit loss rate in Branded cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-Q 2023-Q2*
- "the first quarter of 2023 net credit loss rate in Branded cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-Q 2023-Q1*

**Missing**
- Explicit statement or data for Citi's 2023-Q4 Branded Cards net credit loss rate year-over-year change in a formal disclosure (e.g., 10-Q or 10-K for 2023-Q4).

---
### 🟡 Citi reported fourth-quarter 2023 90+ days past due delinquency rates increased quarter-over-quarter in Branded Cards
**Partially supported**

The 10-Q 2023-Q3 filing explicitly states that the 90+ days past due delinquency rate in Branded Cards increased quarter-over-quarter, but there is no direct evidence from the 10-K 2023-Q4 filing confirming the same trend for Q4 2023. The peer banks' evidence does not directly address Citi's Branded Cards delinquency trend but shows rising net charge-off rates for Citi in Q4 2023, which may imply continued delinquency pressure.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), vs 2019 baseline: missing, peer cross-section: highest among peers (C 1.18% > JPM 0.49% > BAC 0.37%) |

**Supporting**
- "the third quarter of 2023 net credit loss rate and 90+ days past due delinquency rate in Branded cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-Q 2023-Q3*

**Qualifying / watch**
- "The ratios of 90+ days past due and 30–89 days past due are calculated based on EOP loans, net of unearned income." — *Citigroup, 10-K 2023-Q4*

**Missing**
- Explicit 90+ days past due delinquency rate for Branded Cards in 2023-Q4 and its quarter-over-quarter change
- Pre-2020 baseline for Citi's 90+ days past due delinquency rate in Branded Cards

---
### 🟡 Citi reported fourth-quarter 2023 90+ days past due delinquency rates increased year-over-year in Branded Cards
**Partially supported**

The 10-K 2023-Q4 filing for Citigroup does not explicitly state the year-over-year change in 90+ days past due delinquency rates for Branded Cards, but the 10-Q 2023-Q3 filing explicitly states that the 90+ days past due delinquency rate in Branded cards increased year-over-year. Peer banks' evidence shows rising delinquency trends but does not directly address Citi's Branded Cards.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), no pre-2020 baseline available |

**Supporting**
- "the third quarter of 2023 net credit loss rate and 90+ days past due delinquency rate in Branded cards increased quarter-over-quarter and year-over-year" — *Citigroup, 10-Q 2023-Q3*

**Missing**
- Explicit year-over-year 90+ days past due delinquency rate for Branded Cards in 2023-Q4 from Citigroup's 10-K 2023-Q4 filing
- Pre-2020 baseline for Citi's 90+ days past due delinquency rate in Branded Cards

---
### ⚪ Citi reported fourth-quarter 2023 net credit loss rates and 90+ days past due delinquency rates increased quarter-over-quarter and year-over-year in Retail Services
**Insufficient evidence**

The primary evidence provided does not include explicit data or statements on Citi's Retail Services net credit loss rates or 90+ days past due delinquency rates for 2023-Q4, nor their quarter-over-quarter or year-over-year changes. The XBRL metrics confirm Citi's overall net charge-off rate rose QoQ and YoY, but this is not specific to Retail Services. Peer banks' evidence shows rising net charge-off rates but cannot substitute for Citi's Retail Services-specific data. The primary evidence is insufficient to support the claim as stated.

| Ticker | Metric | Period | Value | Change |
|---|---|---|---|---|
| C | net_charge_off_rate | 2023-Q4 | 1.18% | QoQ +0.17pp (rising), YoY trend not explicitly stated but implied by rising QoQ trail |
| C | net_charge_off_rate | 2023-Q3 | 1.01% | QoQ +0.06pp (rising from 2023-Q2) |
| C | net_charge_off_rate | 2023-Q2 | 0.95% | QoQ +0.11pp (rising from 2023-Q1) |
| C | net_charge_off_rate | 2023-Q1 | 0.84% | QoQ +0.11pp (rising from 2022-Q4, inferred from trail) |

**Missing**
- Citi's Retail Services-specific net credit loss rates for 2023-Q4 and prior periods
- Citi's Retail Services-specific 90+ days past due delinquency rates for 2023-Q4 and prior periods
- Explicit quarter-over-quarter and year-over-year changes for Retail Services metrics